# 28 — Triplet Autoencoder WJ 512 (fixed)

Same architecture as nb27 (autoencoder) but trained with **in-batch hard-negative WJ triplet loss + FN filter + reconstruction**,
matching nb15's training protocol exactly. Fixes original: random negatives → hard negatives, added FN masking, batch=2048, 50 epochs.

In [1]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup, eval_recall, l1_simplex, load_dataset,
    nmslib_neighbors, rerank_raw_wj_numpy, save_result,
)

dataset_name = "10k"
out_dim      = 512
device_str   = "cuda:0"
device       = torch.device(device_str if torch.cuda.is_available() else "cpu")
THREADS      = 32
seed         = 42
batch_size   = 2048
epochs       = 50
lr           = 1e-3
weight_decay = 1e-4
max_pos      = 30
margin       = 0.3
lambda_recon = 0.1
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]

METHOD_NAME   = "triplet_autoencoder_wj_512"
NOTEBOOK_NAME = "28_triplet_autoencoder_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_autoencoder_wj_512.pkl"
CKPT_PATH     = "/tmp/best_sota_triplet_autoencoder_wj_512.pt"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"device={device} | batch={batch_size} | epochs={epochs}")


device=cuda:0 | batch=2048 | epochs=50


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [3]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_triplet_loss_inbatch(anchors, positives, margin=0.3,
                             anchor_ids=None, positive_ids=None, gt_matrix=None):
    """In-batch hard-negative WJ triplet loss with FN masking — identical to nb15."""
    sim_ap = wj_sim(anchors, positives)
    mins_c = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    maxs_c = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    sim_cross = mins_c / maxs_c.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    n_fn = 0
    if gt_matrix is not None and anchor_ids is not None:
        fn_mask = gt_matrix[anchor_ids][:, positive_ids].to(sim_cross.device)
        n_fn = int(fn_mask.sum().item())
        if n_fn: sim_cross[fn_mask] = -1e9
    sim_an   = sim_cross.max(dim=1).values
    loss     = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0, n_fn
    return loss[violated].mean(), int(violated.sum().item()), n_fn

class AnchorPositiveDataset(Dataset):
    def __init__(self, vecs_norm, gt_lookup, query_start, max_pos=30):
        self.vecs = torch.tensor(vecs_norm, dtype=torch.float32)
        n = len(vecs_norm)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}")
        if n <= 15_000:
            mat = np.zeros((n, n), dtype=bool)
            for qid, neighbors in gt_lookup.items():
                for nid in neighbors:
                    if nid < n: mat[qid, nid] = True
            self.gt_matrix = torch.from_numpy(mat)
            print(f"GT matrix ({n}x{n}) built")
        else:
            self.gt_matrix = None
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return self.vecs[qid], self.vecs[pid], qid, pid

class TripletAE(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, in_dim, bias=False),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        z = self.encode(x)
        return z, F.relu(self.decoder(z))

def embed_all(model, qt, batch_size=512):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model.encode(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_raw_wj_numpy(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})


In [4]:
model   = TripletAE(qt_norm.shape[1], out_dim).to(device)
dataset = AnchorPositiveDataset(qt_norm, gt, query_start, max_pos=max_pos)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                     num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)
gt_matrix = dataset.gt_matrix
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best = float('inf')
t0_train = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    tot_loss = tot_trip = tot_rec = tot_viol = steps = 0
    for a, p, a_ids, p_ids in loader:
        a = a.to(device, non_blocking=True)
        p = p.to(device, non_blocking=True)
        B = a.shape[0]
        out = model(torch.cat([a, p]))
        za, zp = out[0][:B], out[0][B:]
        rec_a, rec_p = out[1][:B], out[1][B:]
        trip, n_viol, _ = wj_triplet_loss_inbatch(
            za, zp, margin=margin,
            anchor_ids=a_ids.tolist(), positive_ids=p_ids.tolist(),
            gt_matrix=gt_matrix,
        )
        rec_loss = (F.mse_loss(rec_a, a) + F.mse_loss(rec_p, p)) * 0.5
        loss = trip + lambda_recon * rec_loss
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_trip += float(trip.detach())
        tot_rec  += float(rec_loss.detach()); tot_viol += n_viol; steps += 1
    sch.step()
    avg = tot_loss / max(steps, 1)
    if avg < best: best = avg; torch.save(model.state_dict(), CKPT_PATH)
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        elapsed = (time.time() - t0_train) / 60
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | "
              f"trip={tot_trip/steps:.4f} | rec={tot_rec/steps:.6f} | "
              f"viol={tot_viol/steps:.1f} | {elapsed:.1f}min", flush=True)

print(f"best={best:.4f} | saved {CKPT_PATH}")


pairs=46,722
GT matrix (10000x10000) built
epoch 01/50 | loss=0.1830 | trip=0.1829 | rec=0.000926 | viol=1877.4 | 0.3min
epoch 05/50 | loss=0.1366 | trip=0.1366 | rec=0.000000 | viol=1503.0 | 1.6min
epoch 10/50 | loss=0.1263 | trip=0.1263 | rec=0.000000 | viol=1383.1 | 3.1min
epoch 15/50 | loss=0.1191 | trip=0.1191 | rec=0.000000 | viol=1318.4 | 4.6min
epoch 20/50 | loss=0.1131 | trip=0.1131 | rec=0.000000 | viol=1277.2 | 6.1min
epoch 25/50 | loss=0.1077 | trip=0.1077 | rec=0.000000 | viol=1230.5 | 7.6min
epoch 30/50 | loss=0.1030 | trip=0.1030 | rec=0.000000 | viol=1200.0 | 9.2min
epoch 35/50 | loss=0.0990 | trip=0.0990 | rec=0.000000 | viol=1155.5 | 10.8min
epoch 40/50 | loss=0.0951 | trip=0.0951 | rec=0.000000 | viol=1130.8 | 12.4min
epoch 45/50 | loss=0.0933 | trip=0.0933 | rec=0.000000 | viol=1105.5 | 13.9min
epoch 50/50 | loss=0.0930 | trip=0.0930 | rec=0.000000 | viol=1091.7 | 15.5min
best=0.0930 | saved /tmp/best_sota_triplet_autoencoder_wj_512.pt


In [5]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.7114
R@50   = 0.8580
R@100  = 0.8902
R@500  = 0.9774
QPS=10903.2
saved triplet_autoencoder_wj_512 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_autoencoder_wj_512_rerank_500 R@10 = 0.9966
triplet_autoencoder_wj_512_rerank_500 R@50 = 0.9985
triplet_autoencoder_wj_512_rerank_500 R@100 = 0.9987
triplet_autoencoder_wj_512_rerank_500 R@500 = 0.9774
saved triplet_autoencoder_wj_512_rerank_500 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_autoencoder_wj_512_rerank_1000 R@10 = 0.9966
triplet_autoencoder_wj_512_rerank_1000 R@50 = 0.9986
triplet_autoencoder_wj_512_rerank_1000 R@100 = 0.9989
triplet_autoencoder_wj_512_rerank_1000 R@500 = 0.9895
saved triplet_autoencoder_wj_512_rerank_1000 -> /tmp/results_sota_triplet_autoencoder_wj_512.pkl
